# 05 · Fase 5 — Agente integrador, verificação de respostas, Streamlit e LangSmith

Rubricas **5.1 a 5.6**: verificação de consistência factual e coerência lógica, agente único integrando RAG + SQL + API, executor e histórico (equivalências), interface Streamlit e avaliação de qualidade e trajetória.

> **Sobre o modo offline.** O enunciado pede alternativas *dummy* para APIs e credenciais. Sem `OPENAI_API_KEY` (ou com `MODO_OFFLINE=true`), o projeto troca o GPT por um **modelo roteirizado** (`src/llm.py`: regras determinísticas que escolhem a ferramenta e formatam a observação), os embeddings por `DeterministicFakeEmbedding` e as APIs por **fixtures gravados** de chamadas reais (`fixtures/`). A estrutura das chains, do agente, da memória e da verificação é exatamente a mesma; só a "inteligência" muda. As saídas deste notebook foram geradas no modo indicado na célula de configuração.

In [1]:
# Configuração comum: roda a partir da raiz do repositório e mostra o modo de execução
import os
import sys
from importlib.metadata import version
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))
import warnings

warnings.filterwarnings("ignore")

from src import config, llm

print(
    "LangChain",
    version("langchain"),
    "| langchain-core",
    version("langchain-core"),
    "| LangGraph",
    version("langgraph"),
    "| LangSmith",
    version("langsmith"),
)
print(
    "Modo:",
    "OFFLINE (modelo dummy roteirizado + fixtures)" if llm.offline() else f"ONLINE ({config.MODELO_AGENTE})",
)

LangChain 1.4.9 | langchain-core 1.7.6 | LangGraph 1.2.20 | LangSmith 0.14.17
Modo: ONLINE (gpt-4o-mini)


## 5.2 a 5.4 Agente integrador

Um único agente (`montar_agente()`) com cinco categorias de ferramenta: API externa, text-to-SQL, RAG, hardcoded e memória episódica.

> **Notas de equivalência.** (5.3) `create_openai_tools_agent` + `AgentExecutor` foram substituídos por `create_agent`, que compila o mesmo loop de tool calling em um grafo LangGraph. (5.4) `RunnableWithMessageHistory` gera aviso de depreciação; o histórico por sessão é o checkpointer `MemorySaver` com `thread_id`, que dura a sessão inteira (demonstrado abaixo e no Streamlit).

In [2]:
from src.agent import montar_agente, trajetoria

agente = montar_agente()
cfg = {"configurable": {"thread_id": "integrador"}}
perguntas = [
    "Quanto a deputada Tabata Amaral gastou com combustível em 2025?",  # API
    "Compare com a média do partido dela.",  # SQL + memória da sessão
    "A cota pode pagar esse tipo de despesa? Qual o limite em SP?",  # RAG
    "Quem é o fornecedor de CNPJ 43.731.926/0001-60?",  # API (Brasil API)
    "Guarde essa apuração.",  # memória episódica
]
historico = []
for p in perguntas:
    est = agente.invoke({"messages": [("user", p)]}, config=cfg)
    traj = trajetoria(est)
    historico.append((p, est["messages"][-1].content, traj))
    print(
        f"\nUSUÁRIO: {p}\nFERRAMENTAS: {[t['ferramenta'] for t in traj if t['tipo'] == 'decisao']}\nAGENTE: {est['messages'][-1].content[:220]}"
    )


USUÁRIO: Quanto a deputada Tabata Amaral gastou com combustível em 2025?
FERRAMENTAS: ['buscar_deputado', 'listar_despesas']
AGENTE: A deputada Tabata Amaral gastou R$ 48.808,83 com combustíveis em 2025. Fonte: API da Câmara, despesas de 2025.


[AUDITORIA SQL] SELECT ROUND(AVG(t.total),2) AS media FROM (SELECT nome_parlamentar, SUM(valor) AS total FROM despesas_ceap WHERE partido='PSB' AND ano=2025 AND categoria LIKE '%COMBUST%' GROUP BY nome_parlamentar) t LIMIT 200



USUÁRIO: Compare com a média do partido dela.
FERRAMENTAS: ['consultar_ceap', 'calcular_estatisticas']
AGENTE: A média de gastos com combustíveis dos deputados do partido PSB em 2025 foi de R$ 48.849,91. Tabata Amaral gastou 0,08% a menos que a média do partido. Fonte: API da Câmara, despesas de 2025.



USUÁRIO: A cota pode pagar esse tipo de despesa? Qual o limite em SP?
FERRAMENTAS: ['consultar_regras_cota']
AGENTE: Sim, a cota parlamentar pode pagar despesas com combustíveis e lubrificantes, até o limite mensal de R$ 9.392,00 (nove mil trezentos e noventa e dois reais) para deputados em São Paulo. Fonte: Ato da Mesa nº 270, de 19/1



USUÁRIO: Quem é o fornecedor de CNPJ 43.731.926/0001-60?
FERRAMENTAS: ['dados_fornecedor']
AGENTE: O fornecedor do CNPJ 43.731.926/0001-60 é a empresa POSTO BATALHA LTDA, que está ativa e localizada em São Paulo-SP. O CNAE principal é comércio varejista de combustíveis para veículos automotores. A empresa foi aberta e



USUÁRIO: Guarde essa apuração.
FERRAMENTAS: ['gravar_apuracao']
AGENTE: A apuração foi salva com sucesso. Se precisar de mais informações, estou à disposição!


## 5.1 Verificação de consistência factual e coerência lógica

A verificação roda **dentro do agente** (no Streamlit, a cada turno) em três etapas:

1. **Extração estruturada** (`ClaimExtractionList`, Pydantic): cada afirmação numérica vira um *claim* com chave canônica (`gasto_total`, `gasto_categoria`, `media_partido`, `limite_cota_uf`, `artigo_citado`...).
2. **Fact-checking determinístico**: a trajetória do turno diz quais consultas recalcular no SQLite (fonte autoritativa); cada claim é comparado com tolerância de R$ 0,01 → `CONFIRMADO` / `CONTRADITORIO` / `NAO VERIFICADO`.
3. **NLI por LLM** (`LogicalCoherenceReport`): veredito `APOIA` / `REFUTA` / `NEUTRA`, coerência lógica e falácias (erro de cálculo, período trocado, violação de escopo).

In [3]:
from src.verification import consultas_da_trajetoria, verificar_turno

pergunta, resposta, traj = historico[0]
print("consultas da fonte autoritativa:", consultas_da_trajetoria(traj))
v = verificar_turno(resposta, traj)
print("\nfact-check:")
[print("  ", fc) for fc in v["fact_check"]]
print("\nNLI:", v["nli"])

consultas da fonte autoritativa: {'gasto_categoria': "SELECT ROUND(SUM(valor),2) FROM despesas_ceap WHERE id_deputado=204534 AND ano=2025 AND categoria LIKE '%COMBUST%'"}



fact-check:
   {'claim': 'A deputada Tabata Amaral gastou R$ 48.808,83 com combustíveis em 2025.', 'chave': 'gasto_categoria', 'alegado': 48808.83, 'oficial': 48808.83, 'status': 'CONFIRMADO'}

NLI: {'nli_verdict': 'APOIA', 'is_logically_coherent': True, 'detected_fallacies': ['Nenhuma'], 'technical_justification': 'A resposta do agente está em conformidade com os registros oficiais fornecidos. O valor mencionado na resposta (R$ 48.808,83) coincide exatamente com o valor registrado (R$ 48.808,83) para a categoria de gasto. Não há erros de cálculo, pois o valor está correto e dentro da tolerância especificada (0,01). Além disso, não há contradições entre a premissa e a conclusão, nem violações de escopo ou períodos trocados. A fonte citada (API da Câmara, despesas de 2025) é apropriada e relevante para o contexto apresentado.'}


In [4]:
# resposta adulterada: o fact-checking precisa pegar
v_ruim = verificar_turno(
    "Tabata Amaral gastou R$ 99.000,00 com combustível em 2025. Fonte: API da Câmara.", traj
)
print([(fc["claim"], fc["oficial"], fc["status"]) for fc in v_ruim["fact_check"]])

[('Tabata Amaral gastou R$ 99.000,00 com combustível em 2025.', 48808.83, 'CONTRADITORIO')]


## 5.5 Interface Streamlit

`app.py` expõe o agente integrador em um chat: `st.session_state` guarda `thread_id`, mensagens, trajetórias e verificações; `StreamlitCallbackHandler` mostra as chamadas de ferramenta em tempo real; a barra lateral traz a última trajetória e o botão "Nova apuração" (nova thread); abaixo de cada resposta aparece o painel de verificação (fact-check + NLI). Executar com `devbox run app` (ou `streamlit run app.py`).

## 5.6 Avaliação de qualidade e trajetória

**Dataset** (`src/evaluation.py`, exportado em `docs/dataset_avaliacao.json`): seis casos cobrindo API, SQL, RAG, homônimos e fornecedor, cada um com ferramentas esperadas, ferramentas proibidas e valor de referência.

**Avaliadores** (mesma assinatura do `langsmith.evaluate`):
- `trajetoria_correta`: ferramentas esperadas chamadas na ordem certa e nenhuma proibida;
- `valor_correto`: a resposta cita o valor de referência (tolerância de centavos);
- `juiz_llm`: LLM-as-a-judge (CORRETO/INCORRETO). Cuidado herdado da aula 8: `"CORRETO" in veredito` também é verdadeiro para "INCORRETO"; aqui usamos `startswith`. No modo offline o juiz devolve `None` em vez de uma nota fabricada.

In [5]:
import pandas as pd

from src.evaluation import avaliar_localmente, langsmith_disponivel, salvar_dataset_local

print("dataset salvo em", salvar_dataset_local().relative_to(RAIZ))
resultados = pd.DataFrame(avaliar_localmente(agente))
display(resultados)
print(
    "médias:",
    resultados[["trajetoria_correta", "valor_correto", "juiz_llm"]]
    .mean(numeric_only=True)
    .round(2)
    .to_dict(),
)

dataset salvo em docs/dataset_avaliacao.json


[AUDITORIA SQL] SELECT ROUND(AVG(t.total),2) AS media FROM (SELECT nome_parlamentar, SUM(valor) AS total FROM despesas_ceap WHERE partido='PSB' AND ano=2025 AND categoria LIKE '%COMBUST%' GROUP BY nome_parlamentar) t LIMIT 200


,caso,resposta,trajetoria_correta,valor_correto,juiz_llm
0,gasto_categoria_api,"A deputada Tabata Amaral gastou R$ 48.808,83 c...",1,1.0,1
1,gasto_total_api,O gasto total do deputado Nikolas Ferreira em ...,1,1.0,1
2,media_partido_sql,A média de gasto com combustível por deputado ...,1,1.0,1
3,homonimos,Encontrei várias opções de deputados com o nom...,1,NaN,1
4,regra_rag,O limite mensal da cota para um deputado de Sã...,1,1.0,1
5,fornecedor_api,O fornecedor do CNPJ 33.927.528/0001-60 é a em...,1,NaN,1


médias: {'trajetoria_correta': 1.0, 'valor_correto': 1.0, 'juiz_llm': 1.0}


In [6]:
if langsmith_disponivel():
    from src.evaluation import avaliar_no_langsmith, enviar_dataset_langsmith

    ds = enviar_dataset_langsmith()
    print("dataset LangSmith:", ds.name, ds.id)
    exp = avaliar_no_langsmith()
    print(exp)
else:
    print(
        "LANGCHAIN_API_KEY ausente: avaliação registrada localmente acima; no LangSmith será repetida com a chave."
    )

dataset LangSmith: apuracota-casos 6501d508-7202-4e3e-b210-a61d17218b18


View the evaluation results for experiment: 'apuracota-3ed9aadd' at:
https://smith.langchain.com/o/d018734c-9bf1-40b6-b0d7-2c3d5d46f8fe/datasets/6501d508-7202-4e3e-b210-a61d17218b18/compare?selectedSessions=7236f9fc-fb3e-4087-ac9b-271a71d51ef6




[AUDITORIA SQL] SELECT ROUND(AVG(t.total),2) AS media FROM (SELECT nome_parlamentar, SUM(valor) AS total FROM despesas_ceap WHERE partido='PSB' AND ano=2025 AND categoria LIKE '%COMBUST%' GROUP BY nome_parlamentar) t LIMIT 200


<ExperimentResults apuracota-3ed9aadd>


### Análise comparativa no LangSmith

O enunciado pede comparar execuções. O mesmo dataset é avaliado com três configurações, e cada uma vira um experimento no LangSmith:

| experimento | agente | juiz | para que serve |
|---|---|---|---|
| `apuracota-*` | gpt-4o-mini | gpt-4o | configuração de produção (célula acima) |
| `apuracota-dummy-*` | modelo roteirizado, fixtures, embeddings fake | nenhum | piso: o que o código determinístico entrega sem LLM |
| `apuracota-gpt4o-*` | gpt-4o | gpt-4o | teto de capacidade e seu custo |

As variantes são escolhidas alterando `config` em tempo de execução (o agente lê o modelo e o modo offline ao ser montado). Os custos e latências vêm dos traces.

In [7]:
if langsmith_disponivel():
    for rotulo, ajustes in {
        "apuracota-dummy": {"MODO_OFFLINE": True},
        "apuracota-gpt4o": {"MODELO_AGENTE": "gpt-4o"},
    }.items():
        originais = {k: getattr(config, k) for k in ajustes}
        for k, v in ajustes.items():
            setattr(config, k, v)
        try:
            print(rotulo, "->", avaliar_no_langsmith(prefixo=rotulo))
        finally:
            for k, v in originais.items():
                setattr(config, k, v)

[INDICE] salvo com 1536 dims, embeddings offline com 256: refazendo em memoria


View the evaluation results for experiment: 'apuracota-dummy-c29e1208' at:
https://smith.langchain.com/o/d018734c-9bf1-40b6-b0d7-2c3d5d46f8fe/datasets/6501d508-7202-4e3e-b210-a61d17218b18/compare?selectedSessions=72c86389-9587-43a0-a6bf-cfb49d66b322


[AUDITORIA SQL] SELECT ROUND(AVG(t.total),2) AS media_por_deputado FROM (SELECT id_deputado, SUM(valor) AS total FROM despesas_ceap WHERE partido='PSB' AND ano=2025 AND categoria LIKE '%COMBUST%' GROUP BY id_deputado) t LIMIT 200


apuracota-dummy -> <ExperimentResults apuracota-dummy-c29e1208>


View the evaluation results for experiment: 'apuracota-gpt4o-ff8ccc17' at:
https://smith.langchain.com/o/d018734c-9bf1-40b6-b0d7-2c3d5d46f8fe/datasets/6501d508-7202-4e3e-b210-a61d17218b18/compare?selectedSessions=4777030d-f878-4bb8-b515-bb5be7c80de3




[AUDITORIA SQL] SELECT ROUND(AVG(t.total),2) AS media FROM (SELECT nome_parlamentar, SUM(valor) AS total FROM despesas_ceap WHERE partido='PSB' AND ano=2025 AND categoria LIKE '%COMBUST%' GROUP BY nome_parlamentar) t LIMIT 200


apuracota-gpt4o -> <ExperimentResults apuracota-gpt4o-ff8ccc17>


In [8]:
if langsmith_disponivel():
    from src.evaluation import comparar_experimentos

    comparacao = pd.DataFrame(comparar_experimentos())
    display(comparacao.drop(columns=["url"]))
    for _, linha in comparacao.iterrows():
        print(linha["experimento"], "->", linha["url"])

,experimento,modelo,casos,erros,tokens,custo_usd,latencia_media_s,trajetoria_correta,valor_correto,juiz_llm
0,apuracota-ec4abd9d,gpt-4o-mini,6,0,31653,0.0027,3.4,1.00,1.00,1.0
1,apuracota-dummy-10b005df,dummy (offline),6,1,0,0.0000,0.0,0.83,0.75,NaN
2,apuracota-gpt4o-8864075c,gpt-4o,6,0,29630,0.0597,2.6,0.83,0.50,1.0
3,apuracota-3ed9aadd,gpt-4o-mini,6,0,31688,0.0029,4.4,1.00,1.00,1.0
4,apuracota-dummy-c29e1208,dummy (offline),6,0,0,0.0000,0.0,1.00,0.75,NaN
5,apuracota-gpt4o-ff8ccc17,gpt-4o,6,0,29632,0.0522,2.1,0.83,1.00,1.0


apuracota-ec4abd9d -> https://smith.langchain.com/o/d018734c-9bf1-40b6-b0d7-2c3d5d46f8fe/projects/p/e5e2cc8a-6ff9-4141-aeb3-cbcf69f8c193
apuracota-dummy-10b005df -> https://smith.langchain.com/o/d018734c-9bf1-40b6-b0d7-2c3d5d46f8fe/projects/p/6e4edaf4-991b-44a5-ba82-7dacca833dea
apuracota-gpt4o-8864075c -> https://smith.langchain.com/o/d018734c-9bf1-40b6-b0d7-2c3d5d46f8fe/projects/p/e6c4e1a6-24b9-488a-a055-d254b5353980
apuracota-3ed9aadd -> https://smith.langchain.com/o/d018734c-9bf1-40b6-b0d7-2c3d5d46f8fe/projects/p/7236f9fc-fb3e-4087-ac9b-271a71d51ef6
apuracota-dummy-c29e1208 -> https://smith.langchain.com/o/d018734c-9bf1-40b6-b0d7-2c3d5d46f8fe/projects/p/72c86389-9587-43a0-a6bf-cfb49d66b322
apuracota-gpt4o-ff8ccc17 -> https://smith.langchain.com/o/d018734c-9bf1-40b6-b0d7-2c3d5d46f8fe/projects/p/4777030d-f878-4bb8-b515-bb5be7c80de3


**Como ler a comparação.** O dummy acerta trajetória e valor nos casos que suas regras cobrem, a custo zero: é o piso que o código determinístico garante, e mostra que parte da qualidade do agente vem das ferramentas, não do modelo. O gpt-4o-mini é a configuração de produção. O gpt-4o custa cerca de 20 vezes mais por caso e, nas rodadas registradas, não foi melhor: escreveu valores em formato en-US (o que levou o avaliador de valor a ser tornado tolerante a formato, deixando o formato para o juiz) e repetiu o atalho de responder "qual Silva?" sem chamar `buscar_deputado`, que o prompt proíbe. Onde o juiz LLM dá 1,0 e o avaliador determinístico dá 0, a divergência é informação: o juiz aceita o número certo em qualquer formato e aceita a pergunta de esclarecimento como resposta adequada, enquanto a trajetória cobra que a ambiguidade tenha sido verificada na fonte.

**Leitura das métricas.** A trajetória é o indicador mais importante para este agente: uma resposta certa por caminho errado (ex.: média do partido respondida pela API em vez do SQL) seria aprovada por um juiz de texto, mas reprovada por `trajetoria_correta`. `valor_correto` é determinístico e pega erros de cálculo e de período. O juiz LLM entra como terceira opinião; a comparação entre os três é o que permite discutir, no relatório, onde o juiz concorda ou erra. No modo offline, `valor_correto` falha nos casos de SQL e RAG porque o dummy devolve o resultado bruto e os embeddings fake não acham o chunk certo, limitações do mock e não do agente.

## Limitações e próximos passos

- Memória episódica em RAM; persistir em FAISS/SQLite.
- Índice só com o Ato da Mesa; incluir a FAQ da cota (linguagem leiga) e metadados por artigo.
- API da Câmara e CSV divergem; o agente já declara a fonte, mas uma ferramenta de reconciliação seria o passo seguinte.
- Na próxima disciplina (LangGraph): separar planejador, executor e verificador em nós explícitos, com *human-in-the-loop* para homônimos.